In [1]:
!pip install -q transformers numpy accelerate datasets pandas evaluate

In [32]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from datasets import Dataset
from evaluate import load
from scipy.stats import pearsonr
from google.colab import drive


In [47]:
import os
from google.colab import drive

drive.mount('/content/drive')

print(os.listdir('/content/drive/MyDrive/ColabNotebooks'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['learn (1).ipynb', 'learn.ipynb', 'nlp_general_pipeline.ipynb', 'train_data.csv', 'test_data.csv', 'Untitled0.ipynb', 'submission_rogpt2_zeroshot.csv']


In [58]:
train_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/train_data.csv')
test_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/test_data.csv')

In [49]:
MODEL_NAME = "readerbench/RoGPT2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

model.eval()


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: readerbench/RoGPT2-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [70]:
def extract_features(df):
  df = df.copy()
  df["clean_word"] = (
    df["word"].fillna("")
      .str.replace("ţ", "ț")
      .str.replace("ş", "ș")
      .str.replace("Ţ", "Ț")
      .str.replace("Ş", "Ș")
  )

  df["group_key"] = df["word_id"].astype(str).str.rsplit("_", n=1).str[0]
  df["word_idx"] = df["word_id"].astype(str).str.rsplit("_", n=1).str[1].astype(int)

  unique_words = (
    df[["group_key", "word_idx", "clean_word"]]
    .drop_duplicates()
    .sort_values(by=["group_key", "word_idx"])
  )

  surprisal_map = {}

  for group_key, group in unique_words.groupby("group_key"):
    words = group["clean_word"].tolist()
    indices = group["word_idx"].tolist()

    text, spans = "", []
    for w in words:
      spans.append((len(text), len(text) + len(w)))
      text += w + " "

    text = text.strip()

    enc = tokenizer(
        text,
        return_tensors = "pt",
        return_offsets_mapping = True,
        truncation = True,
        max_length = 1024
    )

    input_ids = enc["input_ids"].to(device)
    offsets = enc["offset_mapping"][0].tolist()

    with torch.no_grad():
      logits = model(input_ids = input_ids).logits[0]

      log_probs = torch.log_softmax(logits, dim = -1)

    # surprisal of token = -log P(token | previous token)

    tok_surp = [0.0] + [-log_probs[t - 1, input_ids[0, t]].item() for t in range(1, len(offsets))]

    for w_idx, (w_start, w_end) in zip(indices, spans):
      surprisal_map[(group_key, w_idx)] = sum(
          tok_surp[t] for t, (t_start, t_end) in enumerate(offsets)
          if t_end > w_start and t_start < w_end
      )
  keys = list(zip(df["group_key"], df["word_idx"]))
  df["surprisal"] = [surprisal_map.get(k, 0.0) for k in keys]
  df["word_len"] = df["clean_word"].str.len()
  df["surp_per_char"] = df["surprisal"] / (df["word_len"] + 1)
  df["is_punct"] = df["clean_word"].str.contains(r"[.,;:!?–-]", regex=True).astype(int)
  df["is_upper"] = df["clean_word"].str[0].str.isupper().fillna(False).astype(int)
  df["has_digit_or_url"] = df["clean_word"].str.contains(r"[0-9/]", regex=True).astype(int)

  df = df.sort_values(["participant_id", "group_key", "word_idx"]).reset_index(drop=True)
  df["prev_word_len"] = df.groupby(["participant_id", "group_key"])["word_len"].shift(1).fillna(0)
  df["prev_surprisal"] = df.groupby(["participant_id", "group_key"])["surprisal"].shift(1).fillna(0)
  df["next_word_len"] = df.groupby(["participant_id", "group_key"])["word_len"].shift(-1).fillna(0)
  return df


In [71]:
train_feat = extract_features(train_df)
test_feat = extract_features(test_df)

In [72]:
features = [
    "surprisal", "word_len", "surp_per_char", "is_punct", "is_upper",
    "has_digit_or_url", "word_idx", "prev_word_len", "prev_surprisal", "next_word_len"
]

def eval_metric(y_true, preds):
    r2 = max(0.0, r2_score(y_true, preds, force_finite=True))
    pears = abs(pearsonr(y_true, preds)[0])
    return 100.0 * (pears + r2) / 2.0

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(train_df))

In [73]:
print(train_feat.columns.tolist())

['word_id', 'word', 'answer', 'participant_id', 'text', 'clean_word', 'group_key', 'word_idx', 'surprisal', 'word_len', 'surp_per_char', 'is_punct', 'is_upper', 'has_digit_or_url', 'prev_word_len', 'prev_surprisal', 'next_word_len']


In [74]:
from sklearn.ensemble import HistGradientBoostingRegressor

for tr, val in gkf.split(train_feat, train_feat["answer"], groups=train_feat["participant_id"]):
    reg = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=6, random_state=42)
    reg.fit(train_feat.iloc[tr][features], train_feat.iloc[tr]["answer"])
    oof[val] = reg.predict(train_feat.iloc[val][features])

oof = np.clip(oof, 0, None)

In [75]:
print(f"Local CV Score (must be > 36 to beat baseline): {eval_metric(train_feat['answer'], oof):.2f} / 100")

Local CV Score (must be > 36 to beat baseline): 44.86 / 100


In [76]:
final_model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=6, random_state=42)
final_model.fit(train_feat[features], train_feat["answer"])

HistGradientBoostingRegressor(learning_rate=0.05, max_depth=6, max_iter=300,
                              random_state=42)

In [77]:
test_feat["pred"] = np.clip(final_model.predict(test_feat[features]), 0, None)
test_out = test_df[["datapointID"]].merge(test_feat[["datapointID", "pred"]], on="datapointID", how="left")

In [80]:
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_out["datapointID"],
    "prediction": np.round(test_out["pred"], 10)
})

OUTPUT_DIR = "/content/drive/MyDrive/ColabNotebooks/submission_rogpt2_calibrated.csv"
submission.to_csv(OUTPUT_DIR, index = False)
print(submission.head(10))

   subtaskID  datapointID  prediction
0          1            0  323.092940
1          1            1   42.379311
2          1            2  163.300708
3          1            3   39.692696
4          1            4  505.462052
5          1            5   59.199070
6          1            6  300.143099
7          1            7  194.953877
8          1            8  305.316124
9          1            9  333.661295
